In [4]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [5]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [6]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [7]:
dataset = generate_dataset()

#dataset

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [8]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [9]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # TODO - Grading
    score = 10
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

In [10]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    return results

In [11]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)


results = run_eval(dataset)

In [12]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS S3 Bucket Name Extraction Regex\n\nHere are several solutions depending on your needs:\n\n## Basic Solution\n```regex\ns3://([a-z0-9.-]+)/\n```\n\n## More Robust Solution (Recommended)\n```regex\ns3://([a-z0-9][a-z0-9.-]*[a-z0-9])/\n```\n\n## Most Comprehensive Solution\n```regex\ns3://([a-z0-9](?:[a-z0-9-]*[a-z0-9])?(?:\\.[a-z0-9](?:[a-z0-9-]*[a-z0-9])?)*)/\n```\n\n---\n\n## Implementation Examples\n\n### Python\n```python\nimport re\n\n# Using basic regex\npattern = r's3://([a-z0-9][a-z0-9.-]*[a-z0-9])/'\nuri = \"s3://my-bucket-name/path/to/file.txt\"\nmatch = re.search(pattern, uri)\n\nif match:\n    bucket_name = match.group(1)\n    print(bucket_name)  # Output: my-bucket-name\n```\n\n### JavaScript\n```javascript\nconst pattern = /s3:\\/\\/([a-z0-9][a-z0-9.-]*[a-z0-9])\\//;\nconst uri = \"s3://my-bucket-name/path/to/file.txt\";\nconst match = uri.match(pattern);\n\nif (match) {\n    console.log(match[1]); // Output: my-bucket-name\n}\n```\n\n---\n\n## Re